In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import altair as alt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Enable Altair's data transformer for large datasets
alt.data_transformers.enable('default', max_rows=None)
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [2]:
# load the clustering data for feature selection and clustering steps
engineered_features = pd.read_csv('../../data/clustering/engineered_features.csv')
pre_covid_ef = pd.read_csv('../../data/clustering/pre_covid_engineered_features.csv')
covid_ef = pd.read_csv('../../data/clustering/covid_engineered_features.csv')
post_covid_ef = pd.read_csv('../../data/clustering/post_covid_engineered_features.csv')

eng_feat_non_numeric = ['Ticker', 'Company', 'Industrykey', 'Category']

# Select numeric features for clustering
feature_cols = [col for col in engineered_features.columns if col not in eng_feat_non_numeric]

# apply robust scaling to the features
robust_scaler = RobustScaler()

X_scaled = robust_scaler.fit_transform(engineered_features[feature_cols])
pre_covid_X_scaled = robust_scaler.transform(pre_covid_ef[feature_cols])
covid_X_scaled = robust_scaler.transform(covid_ef[feature_cols])
post_covid_X_scaled = robust_scaler.transform(post_covid_ef[feature_cols])

X_scaled_original = X_scaled.copy()
feature_cols_original = feature_cols.copy()

In [3]:
engineered_features

,Ticker,Company,Industrykey,Category,Return_mean,Return_std,Return_skew,Volume_accum_mean,Volume_accum_std,Volume_accum_max,...,Avg_Volume_Change_4Q_mean,EPS_Volatility_4Q_mean,EPS_Volatility_4Q_max,Avg_EPS_Change_4Q_mean,NetWorth_Volatility_4Q_mean,NetWorth_Volatility_4Q_max,Avg_NetWorth_Change_4Q_mean,NIL_Volatility_4Q_mean,NIL_Volatility_4Q_max,Avg_NIL_Change_4Q_mean
0,A,Agilent Technologies,diagnostics-research,sp500,0.043067,0.117935,-0.036898,4.063654e+07,9.984756e+06,7.083990e+07,...,0.049141,0.981866,3.123564,0.273767,0.037271,0.106995,0.003843,7.080844,28.442236,2.664870
1,AAPL,Apple Inc.,consumer-electronics,sp500,0.071759,0.156266,-0.070810,2.275765e+09,1.038062e+09,6.280072e+09,...,0.014215,0.542979,0.635845,0.133357,0.067775,0.138963,-0.013587,0.544057,0.631385,0.157407
2,ABBV,AbbVie,drug-manufacturers-general,sp500,0.053754,0.125594,0.308907,1.482559e+08,4.964051e+07,3.681825e+08,...,0.059068,1.042537,4.408229,0.440800,0.132883,0.464296,0.052540,1.818792,17.237456,0.794460
3,ABNB,Airbnb,travel-services,sp500,0.008785,0.198991,-0.249100,1.157236e+08,3.252025e+07,2.084769e+08,...,0.012426,7.451058,11.924797,3.378764,0.154001,0.322918,0.076989,5.874528,9.007148,2.637675
4,ABT,Abbott Laboratories,medical-devices,sp500,0.037560,0.097236,-0.021515,1.296637e+08,4.166209e+07,3.113584e+08,...,0.072598,1.016708,3.318321,0.444757,0.111448,0.497696,0.024302,1.018145,3.335739,0.441611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYZ,"Block, Inc.",software-infrastructure,sp500,0.096468,0.327076,0.481598,2.271118e+08,1.048428e+08,5.156862e+08,...,0.166918,2.292477,7.512543,0.634770,0.361052,2.103122,0.180526,2.550951,12.387279,0.768662
499,YUM,Yum! Brands,restaurants,sp500,0.034745,0.108363,-0.532992,4.685435e+07,2.063326e+07,1.166157e+08,...,0.038174,0.811174,1.414547,0.327986,0.384998,3.378038,-0.167517,0.821789,1.418978,0.332638
500,ZBH,Zimmer Biomet,medical-devices,sp500,0.011623,0.137339,-0.521270,3.072515e+07,8.765762e+06,6.129128e+07,...,0.069978,3.484751,21.544848,0.905424,0.052612,0.263689,0.019834,3.341764,19.972112,0.911424
501,ZBRA,Zebra Technologies,communication-equipment,sp500,0.048398,0.202622,0.158789,9.049676e+06,3.469303e+06,2.042610e+07,...,0.070275,1.728278,6.025555,0.088374,0.108103,0.300360,0.038073,2.084955,6.028893,0.180773


In [4]:
# Correlation-Based Feature Clustering for Feature Selection and Redundancy Reduction
distance_threshold = 0.30  # Corresponds to correlation of 0.70 or higher being clustered together

def correlation_feature_reduction(engineered_features, feature_cols, distance_threshold=distance_threshold, examples_to_show=5):
    # Compute Spearman correlation matrix (rank-based, robust to outliers)
    corr_matrix, _ = spearmanr(engineered_features[feature_cols])
    corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)

    # Convert correlation to distance: d = 1 - |r|
    # Features with r=1.0 have distance 0, r=0 have distance 1
    distance_matrix = 1 - np.abs(corr_matrix)

    # Convert square distance matrix to condensed form for linkage
    condensed_dist = squareform(distance_matrix, checks=False)

    # Agglomerative clustering (average linkage)
    linkage_matrix = linkage(condensed_dist, method='average')

    # Cut dendrogram at distance threshold (equiv. to |correlation| >= 1 - threshold)
    feature_clusters = fcluster(linkage_matrix, t=distance_threshold, criterion='distance')

    # Build feature cluster assignments
    feature_cluster_df = pd.DataFrame({
        'Feature': feature_cols,
        'Cluster': feature_clusters,
    })

    # Select one representative per cluster
    # Strategy: pick the feature with highest variance (most information)
    selected_features = []
    cluster_info = []

    for cluster_id in sorted(feature_cluster_df['Cluster'].unique()):
        cluster_members = feature_cluster_df[feature_cluster_df['Cluster'] == cluster_id]['Feature'].tolist()

        # Compute variance for each feature in this cluster
        variances = engineered_features[cluster_members].var()
        best_feature = variances.idxmax()

        selected_features.append(best_feature)
        cluster_info.append({
            'Cluster': cluster_id,
            'Size': len(cluster_members),
            'Representative': best_feature,
            'Members': ', '.join(cluster_members),
        })

    cluster_summary_df = pd.DataFrame(cluster_info)

    print("- Feature Reduction Summary -")
    print(f"Original features:       {len(feature_cols)}")
    print(f"Feature clusters found:  {len(selected_features)}")
    print(f"Reduction:               {len(feature_cols) - len(selected_features)} features removed ({(1 - len(selected_features)/len(feature_cols))*100:.1f}%)")
    print("\nClusters with multiple features (showing redundancy):")
    print(cluster_summary_df[cluster_summary_df['Size'] > 1][['Cluster', 'Size', 'Representative']].to_string(index=False))

    # Show a few example clusters in detail
    print("Example Redundant Feature Clusters:")
    for _, row in cluster_summary_df[cluster_summary_df['Size'] > 1].head(examples_to_show).iterrows():
        print(f"\nCluster {row['Cluster']} ({row['Size']} features) -> Representative: {row['Representative']}")
        members = row['Members'].split(', ')
        for m in members[:5]:
            corr_to_rep = corr_df.loc[row['Representative'], m]
            print(f"  - {m:50s}  (r = {corr_to_rep:+.3f})")
        if len(members) > 5:
            print(f"  ... and {len(members)-5} more")

    return linkage_matrix, corr_df, selected_features, cluster_summary_df


linkage_matrix, corr_df, selected_features, cluster_summary_df = correlation_feature_reduction(
    engineered_features=engineered_features,
    feature_cols=feature_cols,
    distance_threshold=distance_threshold,
    examples_to_show=5,
 )

print('Overall Cluster Summary:')
print(cluster_summary_df)

- Feature Reduction Summary -
Original features:       44
Feature clusters found:  12
Reduction:               32 features removed (72.7%)

Clusters with multiple features (showing redundancy):
 Cluster  Size             Representative
       1     3     Avg_EPS_Change_4Q_mean
       2     3     Avg_NIL_Change_4Q_mean
       3     3       NetWorth_Change_mean
       4     5 NetWorth_Volatility_2Q_max
       5     3           Volume_accum_max
       6     3              MarketCap_max
       8     4          Volume_Change_std
       9    10      EPS_Volatility_2Q_max
      10     3                Return_mean
      11     5          Volatility_2Q_max
Example Redundant Feature Clusters:

Cluster 1 (3 features) -> Representative: Avg_EPS_Change_4Q_mean
  - EPS_Change_mean                                     (r = +0.965)
  - Avg_EPS_Change_2Q_mean                              (r = +0.987)
  - Avg_EPS_Change_4Q_mean                              (r = +1.000)

Cluster 2 (3 features) -> Represen

In [5]:
# Visualization: Feature Dendrogram 
def plot_feature_dendrogram(linkage_matrix, feature_cols, distance_threshold=distance_threshold):
    # Convert scipy linkage to plottable dendrogram data
    dend = dendrogram(linkage_matrix, labels=feature_cols, no_plot=True)

    # Extract dendrogram segments for Altair
    segments = []
    for i, (icoord, dcoord) in enumerate(zip(dend['icoord'], dend['dcoord'])):
        for j in range(len(icoord)-1):
            segments.append({
                'x0': icoord[j], 'y0': dcoord[j],
                'x1': icoord[j+1], 'y1': dcoord[j+1],
                'segment_id': i
            })

    seg_df = pd.DataFrame(segments)

    # Threshold line data
    threshold_df = pd.DataFrame({
        'y': [distance_threshold],
        'label': [f'Threshold = {distance_threshold} (r ≥ {1 - distance_threshold:.2f})']
    })

    # Dendrogram plot
    dendro_base = alt.Chart(seg_df).mark_line(color='steelblue', strokeWidth=1.2)
    dendro_lines = dendro_base.encode(
        x=alt.X('x0:Q', title='Feature Index'),
        x2='x1:Q',
        y=alt.Y('y0:Q', title='Distance (1 - |Spearman r|)', scale=alt.Scale(zero=True)),
        y2='y1:Q',
        detail='segment_id:N'
    )

    threshold_line = alt.Chart(threshold_df).mark_rule(
        color='#E45756', strokeDash=[6, 4], strokeWidth=2
    ).encode(
        y='y:Q',
        tooltip=[alt.Tooltip('label:N', title='Cut Threshold')]
    )

    threshold_label = alt.Chart(threshold_df).mark_text(
        align='left', dx=5, dy=-8, fontSize=11, fontWeight='bold', color='#E45756'
    ).encode(
        x=alt.value(0),
        y='y:Q',
        text='label:N'
    )

    dendro_chart = (dendro_lines + threshold_line + threshold_label).properties(
        title='Feature Dendrogram — Correlation-Based Hierarchical Clustering',
        width=700,
        height=350
    ).configure_axis(
        labelFontSize=11,
        titleFontSize=13
    ).configure_title(
        fontSize=16
    )

    return dendro_chart

dendro_chart = plot_feature_dendrogram(linkage_matrix, feature_cols, distance_threshold)
dendro_chart

alt.LayerChart(...)

In [6]:
# Correlation Heatmap: After Reduction
# Show correlation structure for selected features only
def correlation_heatmap(corr_df, selected_features):
    selected_corr = corr_df.loc[selected_features, selected_features]

    # Prepare data for Altair heatmap
    corr_long = []
    for i, feat1 in enumerate(selected_features):
        for j, feat2 in enumerate(selected_features):
            corr_long.append({
                'Feature1': feat1,
                'Feature2': feat2,
                'Correlation': selected_corr.loc[feat1, feat2],
                'AbsCorr': abs(selected_corr.loc[feat1, feat2])
            })

    corr_long_df = pd.DataFrame(corr_long)

    # Heatmap
    heatmap = alt.Chart(corr_long_df).mark_rect().encode(
        x=alt.X('Feature1:N', title='', axis=alt.Axis(labelAngle=-45, labelLimit=200)),
        y=alt.Y('Feature2:N', title='', axis=alt.Axis(labelLimit=200)),
        color=alt.Color('Correlation:Q', 
                        scale=alt.Scale(scheme='redblue', domain=[-1, 1], reverse=True),
                        title='Spearman r'),
        tooltip=[
            alt.Tooltip('Feature1:N', title='Feature 1'),
            alt.Tooltip('Feature2:N', title='Feature 2'),
            alt.Tooltip('Correlation:Q', format='.3f', title='Correlation')
        ]
    ).properties(
        title=f'Correlation Heatmap — {len(selected_features)} Selected Features (After Reduction)',
        width=600,
        height=600
    ).configure_axis(
        labelFontSize=9,
        titleFontSize=12
    ).configure_title(
        fontSize=15
    )

    return heatmap, selected_corr

heatmap, selected_corr = correlation_heatmap(corr_df, selected_features)
heatmap

alt.Chart(...)

In [7]:
# Create Reduced Feature Sets
selected_feature_idx = [list(feature_cols).index(f) for f in selected_features]

X_reduced_scaled = X_scaled[:, selected_feature_idx]
pre_covid_X_reduced_scaled = pre_covid_X_scaled[:, selected_feature_idx]
covid_X_reduced_scaled = covid_X_scaled[:, selected_feature_idx]
post_covid_X_reduced_scaled = post_covid_X_scaled[:, selected_feature_idx]

In [8]:
# Update Global Feature Set to Use Reduced Features
def update_global_feature_set(feature_cols, X_scaled, selected_features, X_reduced_scaled):
    # Replace with reduced features
    feature_cols = selected_features
    X_scaled = X_reduced_scaled

    print(f"Global feature_cols updated: {len(feature_cols_original)} to {len(feature_cols)} features")
    print(f"X_scaled shape: {X_scaled_original.shape} to {X_scaled.shape}")
    return feature_cols, X_scaled

feature_cols, X_scaled = update_global_feature_set(
    feature_cols, X_scaled, selected_features, X_reduced_scaled
)

Global feature_cols updated: 44 to 12 features
X_scaled shape: (503, 44) to (503, 12)


In [9]:
# Validate Feature Selection Quality
def validate_feature_selection(engineered_features, feature_cols_original, selected_features, cluster_summary_df, corr_df):
    n_orig = len(feature_cols_original)
    n_sel = len(selected_features)

    # 1. Redundancy reduction
    corr_original = corr_df.where(~np.eye(n_orig, dtype=bool)).abs()
    max_corr_before = corr_original.max().max()

    selected_corr_vals = corr_df.loc[selected_features, selected_features]
    max_corr_after = selected_corr_vals.where(~np.eye(n_sel, dtype=bool)).abs().max().max()
    mean_corr_before = corr_original.stack().mean()
    mean_corr_after = selected_corr_vals.where(~np.eye(n_sel, dtype=bool)).abs().stack().mean()

    print("- Feature Selection Validation -\n")
    print(f"{'Features:':<40} {n_orig:>8} -> {n_sel} ({n_sel/n_orig:.0%} kept, {n_orig - n_sel} removed)")
    print(f"\n{'Redundancy (pairwise |Spearman r|):'}")
    print(f"  {'Max correlation:':<38} {max_corr_before:>8.3f} -> {max_corr_after:.3f}")
    print(f"  {'Mean correlation:':<38} {mean_corr_before:>8.3f} -> {mean_corr_after:.3f}")

    # 2. Per-cluster representative quality
    print(f"\n{'Cluster representative quality (variance captured):'}")
    print(f"  {'Cluster':<10} {'Size':<8} {'Representative':<45} {'Var share in cluster':>20}")

    cluster_quality = []
    for _, row in cluster_summary_df.iterrows():
        members = row['Members'].split(', ')
        if len(members) == 1:
            var_share = 1.0
        else:
            variances = engineered_features[members].var()
            total_cluster_var = variances.sum()
            rep_var = variances[row['Representative']]
            var_share = rep_var / total_cluster_var if total_cluster_var > 0 else 1.0
        cluster_quality.append(var_share)
        print(f"  {row['Cluster']:<10} {row['Size']:<8} {row['Representative']:<45} {var_share:>19.1%}")

    mean_var_share = np.mean(cluster_quality)
    print(f"\n  Mean variance share of representative within its cluster: {mean_var_share:.1%}")

    return max_corr_after

max_corr_reduced = validate_feature_selection(
    engineered_features, feature_cols_original, feature_cols, cluster_summary_df, corr_df
)

- Feature Selection Validation -

Features:                                      44 -> 12 (27% kept, 32 removed)

Redundancy (pairwise |Spearman r|):
  Max correlation:                          0.993 -> 0.661
  Mean correlation:                         0.209 -> 0.131

Cluster representative quality (variance captured):
  Cluster    Size     Representative                                Var share in cluster
  1          3        Avg_EPS_Change_4Q_mean                                      33.3%
  2          3        Avg_NIL_Change_4Q_mean                                      33.3%
  3          3        NetWorth_Change_mean                                        33.3%
  4          5        NetWorth_Volatility_2Q_max                                  63.7%
  5          3        Volume_accum_max                                            83.5%
  6          3        MarketCap_max                                               90.0%
  7          1        MarketCap_min                           

In [10]:
## DBSCAN Clustering
# K-Distance Graph
def plot_kdistance_graph(X_scaled, k=3):
    nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
    distances, _ = nbrs.kneighbors(X_scaled)
    distances_sorted = np.sort(distances[:, k - 1], axis=0)

    kdist_df = pd.DataFrame({
        'Index': np.arange(len(distances_sorted)),
        'Distance': distances_sorted
    })

    # eps is at elbow point of k-distance graph, which is where the distance starts to increase sharply
    eps_value = distances_sorted[int(len(distances_sorted) * 0.95)]  # heuristic: 95th percentile distance
    print(f"Optimal eps value based on k-distance graph: {eps_value:.4f}")

    eps_rule_df = pd.DataFrame({'eps': [float(eps_value)]})

    line = alt.Chart(kdist_df).mark_line(color='steelblue').encode(
        x=alt.X('Index:Q', title='Data Points Sorted by Distance'),
        y=alt.Y('Distance:Q', title=f'{k}-Nearest Neighbor Distance', scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip('Index:Q'), alt.Tooltip('Distance:Q', format='.4f')]
    )

    eps_line = alt.Chart(eps_rule_df).mark_rule(
        color='#E45756', strokeDash=[6, 4], strokeWidth=1.8
    ).encode(
        y=alt.Y('eps:Q'),
        tooltip=[alt.Tooltip('eps:Q', title='eps', format='.4f')]
    )

    eps_label = alt.Chart(eps_rule_df).mark_text(
        align='left', dx=6, dy=-8, fontSize=12, fontWeight='bold', color='#E45756'
    ).encode(
        x=alt.value(0),
        y=alt.Y('eps:Q'),
        text=alt.value(f'eps = {float(eps_value):.4f}')
    )

    chart = (line + eps_line + eps_label).properties(
        title=f'K-Distance Graph for DBSCAN Parameter Selection (k={k})',
        width=600,
        height=300
    ).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_title(
        fontSize=18
    )

    return eps_value, chart

eps_value, kdist_chart = plot_kdistance_graph(X_scaled, k=3)
kdist_chart

Optimal eps value based on k-distance graph: 17.0519


alt.LayerChart(...)

In [11]:
def run_dbscan(engineered_features, X_scaled, eps_value, min_samples=2):
    dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
    dbscan_labels = dbscan.fit_predict(X_scaled)

    engineered_features['DBSCAN_Cluster'] = dbscan_labels

    n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)

    print(f"DBSCAN Results  (eps={float(eps_value):.4f}, min_samples={min_samples})")
    print(f"Number of clusters : {n_clusters_dbscan}")
    print(f"Number of noise pts: {n_noise}")

    if n_clusters_dbscan > 1:
        valid_mask = dbscan_labels != -1
        valid_labels = dbscan_labels[valid_mask]
        valid_data = X_scaled[valid_mask]
        if len(set(valid_labels)) > 1:
            dbscan_silhouette = silhouette_score(valid_data, valid_labels)
            print(f"Silhouette Score (excl. noise): {dbscan_silhouette:.3f}")

    # list all tickers labeled as noise
    noise_tickers = engineered_features[engineered_features['DBSCAN_Cluster'] == -1]['Ticker'].tolist()
    print(f"Tickers labeled as noise by DBSCAN: {noise_tickers}")

    return dbscan_labels, noise_tickers

dbscan_labels, noise_tickers = run_dbscan(engineered_features, X_scaled, eps_value, min_samples=2)


DBSCAN Results  (eps=17.0519, min_samples=2)
Number of clusters : 2
Number of noise pts: 23
Silhouette Score (excl. noise): 0.782
Tickers labeled as noise by DBSCAN: ['AAPL', 'ALLE', 'AMCR', 'APP', 'ARES', 'BLDR', 'CART', 'COHR', 'COP', 'DOW', 'GEN', 'HAL', 'IBKR', 'ICE', 'LIN', 'MPC', 'MSFT', 'NVDA', 'SHOP', 'TRGP', 'TSLA', 'VRT', 'WYNN']


In [12]:
# Remove companies with very limited data and outliers in terms of financials
def prepare_clustering_data(df, X_scaled, noise_tickers, selected_features, metadata_cols=['Ticker', 'Company', 'Industrykey', 'Category']):
    # Remove noise tickers
    df_minus_noise = df[~df['Ticker'].isin(noise_tickers)].copy()
    
    # Get indices of companies to keep (same order as in df)
    keep_mask = ~df['Ticker'].isin(noise_tickers)
    
    # Filter scaled features using the same mask
    X_scaled_minus_noise = X_scaled[keep_mask]
    
    # Keep metadata + selected features
    df_selected = df_minus_noise[metadata_cols + selected_features].copy()
    
    return df_minus_noise, df_selected, X_scaled_minus_noise

# show original noise tickers before removal
print(f"Original noise tickers identified by DBSCAN: {noise_tickers}")

# Remove gig_mlms from noise_tickers before filtering -  retained for clustering
# even though DBSCAN labeled them as noise, they are relevant gig-economy companies
gig_mlms = engineered_features[(engineered_features['Category'] == 'gig-work') | (engineered_features['Category'] == 'mlm-work')]['Ticker'].tolist()

for ticker in gig_mlms:
    if ticker in noise_tickers:
        noise_tickers.remove(ticker)
        print(f"'{ticker}' retained: removed from noise list.")

# Apply to all periods
(engineered_features_minus_noise,
selected_features_minus_noise,
reduced_X_scaled_minus_noise) = prepare_clustering_data(engineered_features, X_reduced_scaled, noise_tickers, feature_cols)

(pre_covid_ef_minus_noise,
pre_covid_sf_minus_noise,
pre_covid_X_reduced_minus_noise) = prepare_clustering_data(pre_covid_ef, pre_covid_X_reduced_scaled, noise_tickers, feature_cols)

(covid_ef_minus_noise,
covid_sf_minus_noise,
covid_X_reduced_minus_noise) = prepare_clustering_data(covid_ef, covid_X_reduced_scaled, noise_tickers, feature_cols)

(post_covid_ef_minus_noise, 
 post_covid_sf_minus_noise, 
 post_covid_X_reduced_minus_noise) = prepare_clustering_data(post_covid_ef, post_covid_X_reduced_scaled, noise_tickers, feature_cols)

print(f"Data prepared for {len(selected_features_minus_noise)} companies across all periods")
print(f"Removed {len(noise_tickers)} noise tickers: {noise_tickers}")
print(f"Feature dimensions: {reduced_X_scaled_minus_noise.shape}")


Original noise tickers identified by DBSCAN: ['AAPL', 'ALLE', 'AMCR', 'APP', 'ARES', 'BLDR', 'CART', 'COHR', 'COP', 'DOW', 'GEN', 'HAL', 'IBKR', 'ICE', 'LIN', 'MPC', 'MSFT', 'NVDA', 'SHOP', 'TRGP', 'TSLA', 'VRT', 'WYNN']
'CART' retained: removed from noise list.
'SHOP' retained: removed from noise list.
Data prepared for 482 companies across all periods
Removed 21 noise tickers: ['AAPL', 'ALLE', 'AMCR', 'APP', 'ARES', 'BLDR', 'COHR', 'COP', 'DOW', 'GEN', 'HAL', 'IBKR', 'ICE', 'LIN', 'MPC', 'MSFT', 'NVDA', 'TRGP', 'TSLA', 'VRT', 'WYNN']
Feature dimensions: (482, 12)


In [13]:
# Save clustering data for clustering analysis notebook
# Replace raw feature values with scaled values (ready for clustering)

# Update feature columns with scaled values
selected_features_minus_noise[feature_cols] = reduced_X_scaled_minus_noise
pre_covid_sf_minus_noise[feature_cols] = pre_covid_X_reduced_minus_noise
covid_sf_minus_noise[feature_cols] = covid_X_reduced_minus_noise
post_covid_sf_minus_noise[feature_cols] = post_covid_X_reduced_minus_noise

# Save metadata + scaled features to CSV (ready for clustering)
selected_features_minus_noise.to_csv('../../data/clustering/selected_features_minus_noise.csv', index=False)
pre_covid_sf_minus_noise.to_csv('../../data/clustering/pre_covid_sf_minus_noise.csv', index=False)
covid_sf_minus_noise.to_csv('../../data/clustering/covid_sf_minus_noise.csv', index=False)
post_covid_sf_minus_noise.to_csv('../../data/clustering/post_covid_sf_minus_noise.csv', index=False)

print("Saved metadata + scaled features to CSV files")

selected_features_minus_noise

Saved metadata + scaled features to CSV files


,Ticker,Company,Industrykey,Category,Avg_EPS_Change_4Q_mean,Avg_NIL_Change_4Q_mean,NetWorth_Change_mean,NetWorth_Volatility_2Q_max,Volume_accum_max,MarketCap_max,MarketCap_min,Volume_Change_std,EPS_Volatility_2Q_max,Return_mean,Volatility_2Q_max,Return_skew
0,A,Agilent Technologies,diagnostics-research,sp500,-0.074319,6.700387,-0.405524,-0.225121,-0.212947,0.005373,0.230229,-0.530947,0.032079,0.091116,-0.483583,0.096011
2,ABBV,AbbVie,drug-manufacturers-general,sp500,0.471005,1.365362,0.804841,0.141936,1.103978,5.141489,3.054615,0.113301,0.245468,0.501265,-0.340969,0.625311
3,ABNB,Airbnb,travel-services,sp500,10.062758,6.622816,1.399025,-0.003276,0.396645,1.025774,0.886592,-1.021104,1.295086,-1.224523,0.443060,-0.228790
4,ABT,Abbott Laboratories,medical-devices,sp500,0.483923,0.358919,0.157517,0.212518,0.852305,3.002510,2.335430,0.588628,0.069050,-0.120219,-0.632660,0.119558
5,ACGL,Arch Capital Group,insurance-diversified,sp500,0.213952,0.183678,0.374298,0.012921,-0.229294,-0.103846,-0.372054,0.492001,-0.224344,0.062081,-0.544021,-0.042162
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYZ,"Block, Inc.",software-infrastructure,sp500,1.104271,1.291777,3.952880,1.827585,1.757268,0.930182,-0.429047,1.896908,0.632924,2.140524,2.206236,0.889636
499,YUM,Yum! Brands,restaurants,sp500,0.102694,0.048093,-4.588531,3.138669,-0.010206,0.025074,0.611233,-0.281343,-0.203654,-0.228233,0.202192,-0.663324
500,ZBH,Zimmer Biomet,medical-devices,sp500,1.987892,1.698982,-0.033517,-0.060907,-0.255237,-0.184204,0.371389,0.184212,3.085408,-1.115613,-0.094687,-0.645382
501,ZBRA,Zebra Technologies,communication-equipment,sp500,-0.679580,-0.385075,0.426374,-0.026450,-0.436228,-0.261039,-0.335156,0.079199,0.349598,0.295716,0.494562,0.395537
